# Capítulo V – Inferencia Estadística y Modelos de Regresión en R
## Análisis Cuantitativo con R: Matemáticas, Estadística y Econometría

**Autores del libro:** Daniel Liviano Solís · Maria Pujol Jover
**Editorial:** UOC · Primera edición digital: junio 2017
**Fuente:** Capítulo V, págs. 181-212

---

### Objetivos de aprendizaje
Al finalizar este notebook serás capaz de:
1. Construir intervalos de confianza y contrastes de hipótesis para la media, la varianza y la proporción poblacional.
2. Comparar medias, varianzas y proporciones entre dos grupos.
3. Especificar, estimar (MCO) y diagnosticar modelos de regresión lineal con `lm()`.
4. Aplicar el Análisis de la Varianza (ANOVA) de un factor con `aov()`.

### Dataset utilizado
Se reutilizan las variables demográficas de municipios de Cataluña del Capítulo III:
- `edad`: media de edad de la población en el municipio
- `costa`: variable dicotómica (*Sí/No*) — si el municipio está en la costa
- `t.act`: tasa de actividad en el municipio
- `t.mig`: tasa de inmigración en el municipio

### Cómo usar este notebook
Cada apartado sigue el **mismo número de sección del libro** (p. ej. `1.3` = Inferencia sobre la varianza).

---
# 1. Inferencia sobre parámetros estadísticos (pág. 181-193)

## 1.1. Motivación (pág. 181-182)

La inferencia estadística consiste en analizar o deducir las propiedades de una distribución
subyacente mediante el análisis de una muestra. Se apoya en dos herramientas: los
**intervalos de confianza (IC)** y los **contrastes de hipótesis (CH)**. Esta sección trabaja
sobre los tres parámetros principales: la media poblacional (μ), la varianza (σ²) y la
proporción (π).

> **Las hipótesis nunca se aceptan:** estadísticamente no tiene sentido "aceptar" una
> hipótesis. Solo se rechaza o **no se rechaza** H₀ con la información disponible al
> nivel de confianza dado — el resultado puede cambiar si cambia la muestra o el nivel de confianza.

## 1.2. Inferencia sobre la media poblacional (pág. 182-187)

### 1.2.1. Varianza poblacional desconocida: `t.test()` (pág. 182-185)

Cuando desconocemos la varianza poblacional σ², usamos la distribución **t de Student**
para construir intervalos de confianza (IC) y realizar contrastes de hipótesis (CH) sobre μ:

x̄ ± t_{α/2, n-1} · s/√n

| Argumento | Descripción |
|-----------|-------------|
| `x` | Vector de datos |
| `mu` | Valor hipotético de la media bajo H₀ |
| `alternative` | `'two.sided'` (bilateral, por defecto), `'greater'`, `'less'` |
| `conf.level` | Nivel de confianza (1−α), por defecto `0.95` |

**Regla de decisión:** p-valor ≤ α → se rechaza H₀; p-valor > α → no se rechaza H₀.

In [ ]:
# ---------------------------------------------------------------
# 1.2.1 Contraste bilateral de la media: t.test() (pag. 183)
# H0: mu = 50  vs  H1: mu != 50
# Variable: edad (media de edad de la poblacion municipal)
# Varianza poblacional desconocida -> distribucion t de Student
# t.test() realiza simultaneamente el CH y proporciona el IC
# ---------------------------------------------------------------

t.test(edad, mu=50)

# OUTPUT ESPERADO:
# One Sample t-test
# data:  edad
# t = -51.183, df = 940, p-value < 2.2e-16
# alternative hypothesis: true mean is not equal to 50
# 95 percent confidence interval:
#  42.51746 43.07007
# sample estimates:
# mean of x
#  42.79377

# INTERPRETACION:
# - Estadistico t = -51.183 con 940 grados de libertad
# - p-valor < 2.2e-16 < alpha = 0.05 -> se RECHAZA H0
# - Conclusion: la media de edad NO es 50 anios
# - IC 95%: [42.52 ; 43.07]

### 1.2.2. Varianza poblacional conocida: `z.test()` del paquete PASWR (pág. 185-186)

Cuando conocemos la desviación típica poblacional σ, el IC para μ usa la distribución normal: x̄ ± z_{α/2} · σ/√n. `z.test()` pertenece al paquete **PASWR** y requiere especificar `sigma.x`.

**Contraste planteado** (unilateral por la derecha, confianza 90%): H₀: μ=42 vs H₁: μ>42

In [ ]:
# ---------------------------------------------------------------
# 1.2.2a Instalar y cargar el paquete PASWR (pag. 185)
# ---------------------------------------------------------------
install.packages("PASWR")
library(PASWR)

In [ ]:
# ---------------------------------------------------------------
# 1.2.2b Contraste unilateral (cola derecha) con z.test() (pag. 186)
# H0: mu=42  H1: mu>42
# Varianza poblacional CONOCIDA: sigma.x=20 (desviacion tipica asumida)
# Nivel de confianza: 90% -> alpha = 0.10
# ---------------------------------------------------------------

z.test(edad, alternative='greater', mu=42,
  sigma.x=20, conf.level=0.9)

# OUTPUT ESPERADO:
# One-sample z-Test
# data:  edad
# z = 1.2175, p-value = 0.1117
# alternative hypothesis: true mean is greater than 42
# 90 percent confidence interval:
#  41.95822      Inf
# sample estimates:
# mean of x
#  42.79377

# INTERPRETACION:
# - p-valor = 0.1117 > alpha = 0.10 -> NO se rechaza H0
# - IC unilateral 90%: [41.96 ; +Inf)

### 1.2.3. Diferencia de medias entre muestras independientes (pág. 186-187)

H₀: μᵢ = μc vs H₁: μᵢ ≠ μc (equivalentemente H₀: μᵢ − μc = 0)

En R, `variable ~ factor` dentro de `t.test()` separa la variable en dos submuestras
según los niveles del factor.

> Si se desconocen las varianzas poblacionales → `t.test()` (Welch por defecto).
> Si se conocen → `z.test()` con `sigma.x` y `sigma.y`. Para muestras **apareadas** → `t.test(..., paired = TRUE)`.

In [ ]:
# ---------------------------------------------------------------
# 1.2.3 Contraste bilateral de diferencia de medias: t.test() (pag. 187)
# H0: mu_interior = mu_costa   H1: mu_interior != mu_costa
# Formula edad ~ costa separa edad en dos grupos segun costa
# Varianzas desconocidas -> t de Welch (no asume varianzas iguales)
# ---------------------------------------------------------------

t.test(edad~costa, mu=0)

# OUTPUT ESPERADO:
# Welch Two Sample t-test
# data:  edad by costa
# t = 8.9344, df = 112.83, p-value = 8.97e-15
# alternative hypothesis: true difference in means is not equal to 0
# 95 percent confidence interval:
#  2.194662 3.445345
# sample estimates:
# mean in group No mean in group Si
#         43.00354         40.18354

# INTERPRETACION:
# - p-valor = 8.97e-15 < alpha = 0.05 -> se RECHAZA H0
# - Municipios interiores: media 43.0 anios; costeros: media 40.2 anios

## 1.3. Inferencia sobre la varianza poblacional (pág. 188-189)

### Test F de cociente de varianzas: `var.test()`

Para comparar si las varianzas de dos grupos son iguales se usa la distribución
**F de Snedecor**: F = S₁²/S₂² ~ F_{n-1, m-1}

H₀: σᵢ²/σc² = 1 vs H₁: σᵢ²/σc² ≠ 1

Supuesto clave: ambas muestras provienen de distribuciones **normales independientes**.

In [ ]:
# ---------------------------------------------------------------
# 1.3 Test F de igualdad de varianzas: var.test() (pag. 188-189)
# H0: sigma2_interior / sigma2_costa = 1   H1: ratio != 1
# Bilateral, confianza 90%
# ---------------------------------------------------------------

var.test(edad~costa, ratio=1, conf.level=0.9)

# OUTPUT ESPERADO:
# F test to compare two variances
# data:  edad by costa
# F = 3.519, num df = 870, denom df = 69, p-value = 2.689e-09
# alternative hypothesis: true ratio of variances is not equal to 1
# 90 percent confidence interval:
#  2.571425 4.617085
# sample estimates:
# ratio of variances
#              3.519

# INTERPRETACION:
# - p-valor = 2.689e-09 < alpha = 0.10 -> se RECHAZA H0
# - La varianza en municipios interiores es ~3.5 veces mayor que en los costeros
# - IC 90% del cociente de varianzas: [2.57 ; 4.61]

## 1.4. Inferencia sobre la proporción (pág. 190-193)

### Test de proporciones: `prop.test()`

Cuando la variable de interés es dicotómica, se infiere sobre la proporción poblacional π,
usando la distribución normal: p̂ ± z_{α/2}·√(p̂(1−p̂)/n)

`prop.test()` acepta una muestra (`x`=éxitos, `n`=intentos) o varias muestras (vectores `x` y `n`).
Usa internamente la aproximación Chi-cuadrado con corrección de continuidad.

> **Intervalo de máxima holgura:** si no se conoce p̂, se usa p̂=0.5, que produce el IC más ancho.

In [ ]:
# ---------------------------------------------------------------
# 1.4a Test de proporcion: una sola muestra (pag. 190-191)
# Experimento: lanzamiento de moneda
# H0: pi = 0.5   H1: pi != 0.5 (bilateral, confianza 95%)
# 525 caras en 1000 lanzamientos
# ---------------------------------------------------------------

prop.test(525, 1000, p=0.5)

# OUTPUT ESPERADO:
# 1-sample proportions test with continuity correction
# data:  525 out of 1000, null probability 0.5
# X-squared = 2.401, df = 1, p-value = 0.1213
# alternative hypothesis: true p is not equal to 0.5
# 95 percent confidence interval:
#  0.4935129 0.5562927
# sample estimates:
#     p
# 0.525

# INTERPRETACION:
# - p-valor = 0.1213 > alpha = 0.05 -> NO se rechaza H0
# - No hay evidencia de que la moneda este trucada

In [ ]:
# ---------------------------------------------------------------
# 1.4b Test de proporcion: cuatro muestras simultaneas (pag. 192-193)
# Cuatro series de 1000 lanzamientos cada una
# ---------------------------------------------------------------

# Numero de caras obtenidas en cada una de las 4 series de 1000 lanzamientos
caras <- c(510, 497, 521, 485)

# Vector del tamanio de cada serie: 1000 repetido 4 veces
n <- rep(1000, 4)

# Vector de probabilidades nulas: pi = 0.5 para las 4 series
prob <- rep(0.5, 4)

# Contraste de proporciones para las 4 muestras simultaneamente
prop.test(caras, n, p=prob)

# OUTPUT ESPERADO:
# 4-sample test for given proportions without continuity correction
# data:  caras out of n, null probabilities prob
# X-squared = 3.1, df = 4, p-value = 0.5412
# alternative hypothesis: two.sided
# sample estimates:
#  prop 1  prop 2  prop 3  prop 4
#   0.510   0.497   0.521   0.485

# INTERPRETACION:
# - p-valor = 0.5412 >> alpha = 0.05 -> NO se rechaza H0 en ninguna serie

---
# 2. Modelos de regresión (pág. 194-208)

## 2.1. Motivación (pág. 194-195)

Un modelo de regresión lineal describe la relación entre una variable dependiente y una o
más variables explicativas: yᵢ = xᵢ'β + uᵢ, con E(uᵢ|xᵢ)=0. El objetivo de esta sección es
especificar, estimar por Mínimos Cuadrados Ordinarios (MCO) y diagnosticar ese tipo de modelos en R.

## 2.2. Especificación de modelos (pág. 196-199)

### Fórmulas en R

En R, las especificaciones se almacenan como objetos `formula`: `y ~ x1 + x2 + ...`

| Modelo | Fórmula matemática | Instrucción R |
|--------|-------------------|---------------|
| Aditivo estándar | y = β₀ + β₁x₁ + β₂x₂ | `y ~ x1 + x2` |
| Sin constante | y = β₁x₁ + β₂x₂ | `y ~ 0 + x1 + x2` |
| Regresores sumados | y = β₀ + β₁(x₁+x₂) | `y ~ I(x1 + x2)` |
| x₂ al cuadrado | y = β₀ + β₁x₁ + β₂x₂² | `y ~ x1 + I(x2^2)` |
| Interacción | y = β₀+β₁x₁+β₂x₂+β₃(x₁x₂) | `y ~ x1 * x2` o `y ~ x1+x2+x1:x2` |

> `I()` protege la expresión matemática interior para que R la evalúe aritméticamente y no como sintaxis de fórmula.

In [ ]:
# ---------------------------------------------------------------
# 2.2a Inspeccionar la variable cualitativa 'costa' (pag. 196)
# ---------------------------------------------------------------
class(costa)
# [1] "factor"

levels(costa)
# [1] "No" "Si"

In [ ]:
# ---------------------------------------------------------------
# 2.2b Codificacion manual de variable dicotomica como dummy 0/1 (pag. 197)
# Aunque R puede incluir factores directamente en lm(), aqui se crea
# una variable numerica costa2 para ilustrar el proceso manual.
# ---------------------------------------------------------------

# Paso 1: vector de ceros con la misma longitud que costa
costa2 <- rep(0, length(costa))

# Paso 2: identificar que posiciones corresponden al valor "Si"
si <- which(costa=="Si")

# Paso 3: asignar el valor 1 a las posiciones donde costa es "Si"
costa2[si] <- 1

In [ ]:
# ---------------------------------------------------------------
# 2.2c Especificacion de tres modelos de regresion (pag. 198-199)
# Variable dependiente: edad. Independientes: t.act, t.mig, costa2
# ---------------------------------------------------------------

# MODELO 1: edad = b0 + b1*t.act + b2*t.mig + b3*costa2 + u
e1 <- formula(edad ~ t.act + t.mig + costa2)

# MODELO 2: edad = b0 + b1*t.act + b2*t.mig^2 + u
# I(t.mig^2) protege la operacion para que R la evalue aritmeticamente
e2 <- formula(edad ~ t.act + I(t.mig^2))

# MODELO 3: edad = b0 + b1*t.mig + b2*costa2 + b3*(t.mig*costa2) + u
# y~a*b expande a y~a+b+a:b (termino de interaccion)
e3 <- formula(edad ~ t.mig * costa2)

## 2.3. Estimación de modelos (pág. 200-204)

### Mínimos Cuadrados Ordinarios (MCO): `lm()`

MCO minimiza la suma de cuadrados de los residuos eᵢ = yᵢ − ŷᵢ. `lm()` estima los parámetros del modelo.

| Argumento | Descripción |
|-----------|-------------|
| `formula` | Especificación del modelo |
| `data` | Base de datos (opcional si las variables están en el workspace) |
| `subset` | Condición lógica para estimar con un subconjunto de los datos |

In [ ]:
# ---------------------------------------------------------------
# 2.3a Estimacion de los tres modelos por MCO: lm() (pag. 200)
# ---------------------------------------------------------------
m1 <- lm(e1)   # Modelo 1: aditivo con t.act, t.mig, costa2
m2 <- lm(e2)   # Modelo 2: con t.act y t.mig cuadrado
m3 <- lm(e3)   # Modelo 3: con interaccion t.mig * costa2

In [ ]:
# ---------------------------------------------------------------
# 2.3b Sumario completo del modelo 1: summary() (pag. 200-201)
# ---------------------------------------------------------------
summary(m1)

# OUTPUT ESPERADO:
# Call:
# lm(formula = e1)
#
# Residuals:
#     Min      1Q  Median      3Q     Max
# -10.836  -2.834  -0.074   2.661  15.585
#
# Coefficients:
#              Estimate Std.Error t value Pr(>|t|)
# (Intercept)    45.42     0.259  175.201  < 2e-16 ***
# t.act          -0.04     0.005   -8.367  < 2e-16 ***
# t.mig          -0.14     0.018   -7.423 2.57e-13 ***
# costa2         -1.52     0.532   -2.867  0.00424 **
# ---
# Residual standard error: 3.998 on 937 degrees of freedom
# Multiple R-squared:  0.1457
# Adjusted R-squared:  0.1429
# F-statistic: 53.25 on 3 and 937 DF,  p-value: < 2.2e-16

# INTERPRETACION:
# - Todos los coeficientes son significativos al 1%
# - R2 = 0.146: el modelo explica el 14.6% de la variabilidad de edad

In [ ]:
# ---------------------------------------------------------------
# 2.3c Extraccion de coeficientes: coefficients() (pag. 202)
# ---------------------------------------------------------------

# Solo los coeficientes estimados (beta_hat)
coefficients(m1)
# (Intercept)        t.act        t.mig       costa2
#  45.42711338 -0.04963013 -0.14049617 -1.52637227

# Tabla completa: coeficientes + errores estandar + t + p-valor
coefficients(summary(m1))
#              Estimate Std.Error   t value    Pr(>|t|)
# (Intercept) 45.4271    0.2592  175.2008  0.00e+00
# t.act       -0.0496    0.0059   -8.3673  2.12e-16
# t.mig       -0.1404    0.0189   -7.4228  2.57e-13
# costa2      -1.5263    0.5324   -2.8669  4.23e-03

In [ ]:
# ---------------------------------------------------------------
# 2.3d Intervalos de confianza de los coeficientes: confint() (pag. 203)
# El IC de beta es: beta_hat +/- t_{alpha/2, n-k} * SE(beta_hat)
# level=0.9: IC al 90% de confianza
# ---------------------------------------------------------------
confint(m1, level=0.9)
#                    5 %        95 %
# (Intercept) 45.0002  45.8540
# t.act       -0.0593  -0.0398
# t.mig       -0.1716  -0.1093
# costaSi     -2.4029  -0.6497

# INTERPRETACION: ningun IC contiene el 0 -> todos los coeficientes
# son significativos al 90%

In [ ]:
# ---------------------------------------------------------------
# 2.3e Estimacion con subconjunto de datos: subset= (pag. 204)
# Solo municipios costeros (costa2==1) con t.mig > 15%
# ---------------------------------------------------------------
summary(lm(e2, subset=(costa2==1 & t.mig >15)))

# OUTPUT ESPERADO:
# Coefficients:
#              Estimate Std.Error t value Pr(>|t|)
# (Intercept) 41.6038    0.9166  45.386  <2e-16 ***
# t.act       -0.0428    0.0339  -1.263   0.213
# I(t.mig^2)  -0.0008    0.0006  -1.200   0.237
# Residual standard error: 2.207 on 43 degrees of freedom
# Multiple R-squared:  0.05629,   Adjusted R-squared:  0.0124

# INTERPRETACION: en la submuestra, solo el intercepto es
# significativo; R2 muy bajo (0.056)

## 2.4. Diagnóstico de modelos (pág. 205-208)

Al aplicar `plot()` sobre un objeto `lm`, R genera automáticamente **4 gráficos de diagnóstico**:

1. **Residuals vs Fitted:** detecta tendencias y heteroscedasticidad.
2. **Normal Q-Q:** cuantiles empíricos vs. teóricos normales; deben seguir la bisectriz.
3. **Scale-Location:** √|eᵢ| estandarizados; ayuda a ver si la varianza es constante.
4. **Residuals vs Leverage:** detecta observaciones influyentes.

> **Criterio de comparación de modelos:** con la misma variable dependiente, se prefiere el modelo con mayor **R² ajustada**.

In [ ]:
# ---------------------------------------------------------------
# 2.4 Graficos de diagnostico del modelo 1: plot(m1) (pag. 205-208)
# ---------------------------------------------------------------

# Ajustar el layout para mostrar los 4 graficos en una cuadricula 2x2
par(mfrow = c(2, 2))

# Generar los 4 graficos de diagnostico del modelo estimado m1
plot(m1)

# Restaurar el layout original
par(mfrow = c(1, 1))

---
# 3. Análisis de la Varianza - ANOVA (pág. 209-211)

### ANOVA de un factor: `aov()`

ANOVA descompone la variación total observada: SCT = SCE + SCD

| Componente | Significado |
|------------|-------------|
| SCT | Suma de cuadrados total |
| SCE | Variación explicada por los grupos |
| SCD | Variación no explicada (dentro de grupos) |

Estadístico: F* = [SCE/(k−1)] / [SCD/(N−k)] ~ F_{α, k−1, N−k}

**Contraste:** H₀: μc = μᵢ vs H₁: H₀ no es cierta

`aov()` es análoga a `lm()`. Supone grupos de distribuciones normales independientes con varianzas iguales.

In [ ]:
# ---------------------------------------------------------------
# 3 ANOVA de un factor: aov() (pag. 209-211)
# Descompone la variabilidad de 'edad' segun el factor 'costa'
# H0: mu_costa = mu_interior  vs  H1: H0 no es cierta
# ---------------------------------------------------------------

# Estimacion del modelo ANOVA
a1 <- aov(edad~costa)

# Tabla ANOVA completa
summary(a1)

# OUTPUT ESPERADO:
#            Df Sum Sq Mean Sq F value    Pr(>F)
# costa        1    515   515.3   28.43 1.22e-07 ***
# Residuals  939  17019    18.1

# INTERPRETACION:
# - p-valor = 1.22e-07 << alpha = 0.05 -> se RECHAZA H0
# - La media de edad en municipios costeros SI es diferente de la
#   de los municipios de interior (coincide con t.test(edad~costa))

---
## Resumen de funciones del capítulo

| Función | Paquete | Propósito |
|---------|---------|----------|
| `t.test(x, mu, alternative, conf.level)` | base | Contraste e IC para la media (varianza desconocida) |
| `z.test(x, mu, sigma.x, alternative, conf.level)` | PASWR | Contraste e IC para la media (varianza conocida) |
| `var.test(x~g, ratio, conf.level)` | base | Contraste de igualdad de varianzas (test F) |
| `prop.test(x, n, p, alternative)` | base | Contraste e IC para proporciones |
| `formula(y ~ x1 + x2)` | base | Especificación de modelos |
| `lm(formula, data, subset)` | base | Estimación MCO de regresión lineal |
| `summary(modelo)` | base | Sumario completo del modelo estimado |
| `coefficients(modelo)` | base | Extracción de coeficientes |
| `confint(modelo, level)` | base | IC de los coeficientes |
| `plot(modelo)` | base | Gráficos de diagnóstico (4 paneles) |
| `aov(formula)` | base | ANOVA de uno o más factores |

## Conclusión

Este notebook cubre el flujo completo de inferencia estadística y modelado de regresión en R
del **Capítulo V**, siguiendo el mismo orden y numeración de secciones que el libro:

1. **Inferencia sobre parámetros (1.2-1.4):** `t.test()`, `z.test()`, `var.test()` y `prop.test()`
   siguen una estructura uniforme: estadístico de contraste, p-valor, IC y estimaciones muestrales.
2. **Regresión lineal (2.2-2.4):** el flujo es *especificación* (`formula`) → *estimación* (`lm()`)
   → *diagnóstico* (`summary()`, `confint()`, `plot()`).
3. **ANOVA (3):** `aov()` extiende la regresión al análisis de varianza.
4. **Decisión estadística:** en todos los contrastes, comparar el p-valor con α — las
   hipótesis no se aceptan, solo se deja de rechazarlas.

### Cómo ejecutar
1. Requiere las variables `edad`, `costa`, `t.act`, `t.mig` del Capítulo III (ver
   `Cap03_Analisis_Datos_Estadistica_Descriptiva_R.ipynb`).
2. Abrir en **Jupyter Lab**, **VS Code** o **Google Colab** con kernel `R`.
3. La sección 1.2.2 requiere el paquete `PASWR` (se instala automáticamente).

---
*Notebook estandarizado a partir del PDF oficial del capítulo (Cap05_Inferencia_Estadistica_R.pdf).*
*Referencia: Liviano Solís, D. & Pujol Jover, M. (2017). Análisis cuantitativo con R: matemáticas, estadística y econometría. Editorial UOC.*